# ДЗ 7. Срезы orders.csv: groupby, pivot_table, merge
Срок: **до воскресенья 29.11, 24:00**.
Сдача: папка `hw07` репозитория `dsa-homework` (`hw07.ipynb`): `git add hw07` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать. Рядом с каждой долей или средним указывайте, на скольких заказах оно посчитано.

Инструменты: всё с уроков 1–7. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
t = pd.DataFrame({"c": ["a", "b", "a"], "x": [1, 2, 3]})
print(t.groupby("c")["x"].sum().tolist())
# 2
t = pd.DataFrame({"c": ["a", "a", "b"], "d": [1, None, 3]})
print(t.groupby("c")["d"].count().tolist(), t.groupby("c").size().tolist())
# 3
t = pd.DataFrame({"c": ["a", "a", "b"], "m": [9, 9, 10], "x": [10, 30, 5]})
print(t.pivot_table(index="c", columns="m", values="x").loc["a", 9])
# 4
a = pd.DataFrame({"id": [1, 2, 3]})
b = pd.DataFrame({"id": [1, 2], "ch": ["x", "y"]})
print(len(a.merge(b, on="id")), len(a.merge(b, on="id", how="left")))
# 5
b = pd.DataFrame({"id": [1, 1, 2], "ch": ["x", "x", "y"]})
print(len(a.merge(b, on="id", how="left")))
```

Ваши ответы и объяснения:

1. `[4, 2]`. У группы `a` значения 1 и 3, в сумме 4, у `b` одно значение 2. Группы идут по алфавиту.
2. `[1, 1] [2, 1]`. `count` не считает пропуски, поэтому у `a` только 1 значение из двух. `size` считает все строки группы: у `a` их 2, у `b` 1.
3. `20.0`. Без `aggfunc` pivot_table считает среднее: у `a` в месяце 9 значения 10 и 30, среднее 20.0.
4. `2 3`. `inner` оставляет только id, которые есть в обеих таблицах (1 и 2). `left` сохраняет все строки `a`, у id 3 просто будет NaN.
5. `4`. В `b` id 1 записан дважды, поэтому строка с id 1 из `a` размножается на две. Плюс id 2 и id 3, всего 4.

In [1]:
# проверка предсказаний
import pandas as pd

t = pd.DataFrame({"c": ["a", "b", "a"], "x": [1, 2, 3]})
print(t.groupby("c")["x"].sum().tolist())
t = pd.DataFrame({"c": ["a", "a", "b"], "d": [1, None, 3]})
print(t.groupby("c")["d"].count().tolist(), t.groupby("c").size().tolist())
t = pd.DataFrame({"c": ["a", "a", "b"], "m": [9, 9, 10], "x": [10, 30, 5]})
print(t.pivot_table(index="c", columns="m", values="x").loc["a", 9])
a = pd.DataFrame({"id": [1, 2, 3]})
b = pd.DataFrame({"id": [1, 2], "ch": ["x", "y"]})
print(len(a.merge(b, on="id")), len(a.merge(b, on="id", how="left")))
b = pd.DataFrame({"id": [1, 1, 2], "ch": ["x", "x", "y"]})
print(len(a.merge(b, on="id", how="left")))

[4, 2]
[1, 1] [2, 1]
20.0
2 3
4


In [2]:
# Загрузка orders.csv и customers.csv: в Colab нажмите «Выбрать файлы» и укажите оба файла из LMS.
# Если файлы уже лежат рядом с ноутбуком, загрузка пропускается.
import os
if not (os.path.exists("orders.csv") and os.path.exists("customers.csv")):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv и customers.csv в одну папку с ноутбуком")
print("orders.csv:", os.path.exists("orders.csv"), "| customers.csv:", os.path.exists("customers.csv"))

orders.csv: True | customers.csv: True


In [3]:
# Очистка урока 6 одной ячейкой: суммы, города, дубли, даты, TEST
import pandas as pd
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

df = pd.read_csv("orders.csv")
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
df["city"] = df["city"].str.strip().str.lower().map(CITY)
df = df.drop_duplicates()
df["order_date"] = pd.to_datetime(df["order_date"])
df["month"] = df["order_date"].dt.month
df = df[df["customer_id"] != "TEST"].copy()    # copy: дальше добавим столбцы
df.shape

(5280, 9)

In [4]:
df["cancel"] = df["status"] == "отменён"
paid = df[df["status"] == "оплачен"]
cu = pd.read_csv("customers.csv")
len(paid), cu.shape

(4739, (2200, 4))

Можно вместо ячейки очистки взять свою функцию `clean()` из ДЗ 6. Проверка: 5 280 строк, 4 739 оплаченных.

## [База] 1. Сводка по городам одной таблицей
Через `agg` с именами: число заказов, доля отмен, средний срок доставки по городам. Добавьте столбец «выручка оплаченных»: `by_city["revenue"] = paid.groupby("city")["amount"].sum()` (Pandas выровняет по ярлыкам, урок 5). Отсортируйте по выручке.
Какой город выделяется сразу по двум показателям?

In [5]:
# 1. сводка по городам
by_city = df.groupby("city").agg(orders=("order_id", "count"),
                                 cancel_share=("cancel", "mean"),
                                 avg_days=("delivery_days", "mean"))
by_city["revenue"] = paid.groupby("city")["amount"].sum()
by_city.sort_values("revenue", ascending=False).round(3)

,orders,cancel_share,avg_days,revenue
city,,,,
Алматы,2116,0.078,1.997,29144090.0
Астана,1339,0.088,2.019,17884750.0
Шымкент,776,0.206,4.451,8602660.0
Караганда,519,0.091,1.941,7020200.0
Актобе,387,0.106,2.070,5117380.0
Атырау,143,0.077,2.036,1893850.0


**Ответ:** больше всего выручки у Алматы (29,1 млн ₸, 2 116 заказов), потом Астана (17,9 млн, 1 339) и Шымкент (8,6 млн, 776). Шымкент выделяется сразу по двум показателям: отменяют там 20,6% заказов, а в остальных городах 7,7-10,6%. И доставка в среднем 4,45 дня, когда в других городах около 2.

**Менеджеру:** по выручке Шымкент третий, но это единственный город, где плохо и с отменами, и со сроками. Смотреть надо туда.

## [База] 2. ДЗ 4 одной строкой
В ДЗ 4 вы считали категории циклами и NumPy. Повторите через `pivot_table`: выручка оплаченных по категориям и месяцам, изменение в %, средний чек по категориям и месяцам.
Сверьте с ответами ДЗ 4: лидер Электроника 34,4 млн ₸, Одежда −7,2%, средний чек Электроники в октябре 33 600 ₸. Сколько строк кода было и сколько стало?

In [6]:
# 2. ДЗ 4 через pivot_table
cat_rev = paid.pivot_table(index="category", columns="month", values="amount", aggfunc="sum")
cat_rev["total"] = cat_rev[9] + cat_rev[10]
cat_rev["change_%"] = (cat_rev[10] / cat_rev[9] - 1) * 100
cat_check = paid.pivot_table(index="category", columns="month", values="amount", aggfunc="mean")

print(cat_rev.sort_values("total", ascending=False).round(1))
print("\nсредний чек:")
print(cat_check.round(0))
print("\nна скольких оплаченных заказах:")
print(paid.pivot_table(index="category", columns="month", values="amount", aggfunc="count"))

month                 9          10       total  change_%
category                                                 
Электроника  17402330.0  16968100.0  34370430.0      -2.5
Одежда        9167100.0   8505730.0  17672830.0      -7.2
Дом           4997730.0   5125790.0  10123520.0       2.6
Красота       2610950.0   2678990.0   5289940.0       2.6
Книги         1065300.0   1140910.0   2206210.0       7.1

средний чек:
month             9        10
category                     
Дом          10499.0  10768.0
Книги         4020.0   3816.0
Красота       6187.0   6159.0
Одежда       13521.0  13046.0
Электроника  32773.0  33600.0

на скольких оплаченных заказах:
month         9    10
category             
Дом          476  476
Книги        265  299
Красота      422  435
Одежда       678  652
Электроника  531  505


**Ответ:** всё сошлось с ДЗ 4. Лидер Электроника, 34,4 млн ₸ за два месяца (1 036 оплаченных заказов). Сильнее всех упала Одежда, -7,2%. Средний чек Электроники в октябре 33 600 ₸ на 505 заказах.

В ДЗ 4 на эти две задачи ушло 19 строк кода, и это не считая загрузки в отдельные массивы. Здесь сами расчёты занимают 4 строки, остальное просто печать.

## [База] 3. Отмены и срок доставки
По `delivery_days`: число заказов и доля отмен. Во сколько раз отменяют чаще при доставке за 8 дней, чем за 1 день?
Что это значит для Шымкента, где в октябре доставка 5–8 дней?

In [7]:
# 3. отмены по сроку доставки
by_days = df.groupby("delivery_days").agg(orders=("order_id", "count"),
                                          cancel_share=("cancel", "mean"))
print(by_days.round(3))

ratio = by_days.loc[8, "cancel_share"] / by_days.loc[1, "cancel_share"]
print(f"\nпри доставке за 8 дней отменяют в {ratio:.1f} раза чаще, чем за 1 день")

               orders  cancel_share
delivery_days                      
1.0              1557         0.042
2.0              1570         0.080
3.0              1590         0.134
5.0                86         0.221
6.0               120         0.217
7.0                95         0.347
8.0               104         0.433

при доставке за 8 дней отменяют в 10.2 раза чаще, чем за 1 день


**Ответ:** чем дольше доставка, тем больше отмен. За 1 день отменяют 4,2% заказов (из 1 557), за 3 дня уже 13,4% (из 1 590), а за 8 дней 43,3% (из 104). Разница между 8 днями и 1 днём в 10,2 раза.

Для Шымкента это значит, что при октябрьских 5-8 днях доставки стоит ждать 22-43% отмен. Так и вышло: в октябре там отменили 30,7% заказов.

## [Стандарт] 4. Исправьте код от AI
AI посчитал изменение выручки по городам и пишет менеджеру: «В Шымкенте выручка выросла на 2,4%, проблем нет». В коде **две** ошибки. Найдите их, исправьте и напишите менеджеру правильную цифру.
```python
p = df.pivot_table(index="city", columns="month", values="amount")
p["chg"] = p[10] / p[9] - 1
print(p["chg"].round(3))
```

In [8]:
# 4. код из условия
p = df.pivot_table(index="city", columns="month", values="amount")
p["chg"] = p[10] / p[9] - 1
print("как в коде из условия, Шымкент:", p.loc["Шымкент", "chg"].round(3))

# ошибка 1: нет aggfunc, а по умолчанию это среднее, то есть средний чек вместо выручки
# ошибка 2: взят весь df, в выручку попали отменённые заказы, нужен paid
p = paid.pivot_table(index="city", columns="month", values="amount", aggfunc="sum")
p["chg"] = p[10] / p[9] - 1
print(p["chg"].sort_values().round(3))

как в коде из условия, Шымкент: 0.024
city
Шымкент     -0.113
Караганда   -0.076
Астана      -0.020
Актобе      -0.014
Алматы       0.010
Атырау       0.050
Name: chg, dtype: float64


**Ответ:** первая ошибка: в `pivot_table` не указан `aggfunc`, а по умолчанию это среднее. Получился средний чек, а не выручка. Вторая: таблица построена по всему `df`, вместе с отменёнными заказами, а выручку надо считать только по оплаченным. Вместе они и дают +2,4%.

**Менеджеру:** в Шымкенте выручка не выросла, а упала на 11,3%, это худший результат среди всех городов. Проблема есть.

## [Стандарт] 5. Возраст клиентов
Соедините заказы с клиентами (`merge`, `how="left"`, проверьте `len`). Возрастная группа: `m["age_group"] = m["age"] // 10 * 10`.
По группам: число заказов, доля отмен, средний и медианный чек оплаченных.
AI предлагает: «Группа 60 покупает дороже всех и чаще отменяет, сделаем для неё отдельную акцию». Какие возрасты попали в группу 60 и сколько в ней заказов? Подтверждает ли медиана вывод AI?

In [9]:
# 5. возраст клиентов
m = df.merge(cu, on="customer_id", how="left")
print("строк до и после merge:", len(df), len(m), "| без возраста:", m["age"].isna().sum())

m["age_group"] = m["age"] // 10 * 10
m_paid = m[m["status"] == "оплачен"]

by_age = m.groupby("age_group").agg(orders=("order_id", "count"), cancel_share=("cancel", "mean"))
by_age["paid_orders"] = m_paid.groupby("age_group").size()
by_age["mean_check"] = m_paid.groupby("age_group")["amount"].mean()
by_age["median_check"] = m_paid.groupby("age_group")["amount"].median()
print(by_age.round(3))

g60 = m[m["age_group"] == 60]
print("\nвозрасты в группе 60:", g60["age"].unique().tolist(),
      "| клиентов:", g60["customer_id"].nunique(), "| заказов:", len(g60))
print("возраст в customers.csv: от", cu["age"].min(), "до", cu["age"].max())
print("\nсамые дорогие оплаченные заказы группы 60:")
print(m_paid.loc[m_paid["age_group"] == 60, "amount"].sort_values(ascending=False).head(5).tolist())

строк до и после merge: 5280 5280 | без возраста: 0
           orders  cancel_share  paid_orders  mean_check  median_check
age_group                                                             
10            229         0.087          209   14044.737       10700.0
20           1226         0.096         1108   14130.415       10100.0
30           1241         0.111         1103   15142.339       10670.0
40           1170         0.111         1040   14792.971       10050.0
50           1294         0.092         1175   14671.796       10240.0
60            120         0.133          104   16779.135       10495.0

возрасты в группе 60: [60] | клиентов: 46 | заказов: 120
возраст в customers.csv: от 18 до 60

самые дорогие оплаченные заказы группы 60:
[71880.0, 63720.0, 62050.0, 51430.0, 46800.0]


**Ответ:** после `merge` строк столько же, 5 280, и у всех заказов нашёлся возраст. Группа 60 на самом деле состоит только из 60-летних: старше 60 в базе никого нет. Это 46 клиентов и 120 заказов (104 оплаченных), самая маленькая группа после 10 (там только 18-19 лет).

Средний чек у группы 60 правда самый высокий, 16 779 ₸. Но медиана 10 495 ₸, как у всех остальных (10 050-10 700 ₸), то есть среднее тянут вверх несколько дорогих заказов электроники по 60-70 тысяч. Отмен 13,3%, это 16 заказов из 120, при 9-11% у других групп разница в пару-тройку заказов.

**Менеджеру:** вывод про отдельную акцию для группы 60 данными не подтверждается. Покупают они так же, как все, а разница в среднем и в отменах получилась из-за маленькой группы.

## [Стандарт] 6. Повторные покупки
Сколько заказов у каждого клиента (`groupby("customer_id").size()`)? Сколько клиентов сделали 1, 2, 3… заказа (`value_counts().sort_index()`)? Какая доля клиентов купила больше одного раза?
Сколько клиентов заказывали и в сентябре, и в октябре (`groupby("customer_id")["month"].nunique()`)? Сколько клиентов из customers.csv не заказали ничего?

In [10]:
# 6. повторные покупки
per_client = df.groupby("customer_id").size()
print("клиентов с заказами:", len(per_client))
print(per_client.value_counts().sort_index())
print(f"больше одного заказа: {(per_client > 1).mean():.1%}")

months_per_client = df.groupby("customer_id")["month"].nunique()
both = (months_per_client == 2).sum()
print(f"\nзаказывали и в сентябре, и в октябре: {both} ({both / len(per_client):.1%})")

no_orders = ~cu["customer_id"].isin(df["customer_id"])
print("клиентов из customers.csv без заказов:", no_orders.sum(), "из", len(cu))

клиентов с заказами: 2002
1     501
2     563
3     434
4     293
5     127
6      58
7      15
8       9
9       1
10      1
Name: count, dtype: int64
больше одного заказа: 75.0%

заказывали и в сентябре, и в октябре: 1064 (53.1%)
клиентов из customers.csv без заказов: 198 из 2200


**Ответ:** заказы сделали 2 002 клиента. Один раз купили 501 человек, а 75,0% клиентов заказывали больше одного раза, рекордсмен 10 раз. И в сентябре, и в октябре заказывали 1 064 клиента, это 53,1% всех покупателей. Ещё 198 клиентов из 2 200 в customers.csv за эти два месяца не заказали ничего.

**Менеджеру:** база в основном возвращается, три четверти клиентов покупают повторно. Отдельно можно поработать с 198 клиентами, которые зарегистрированы, но ничего не заказали.

## [Челлендж] 7. Сколько стоил сбой доставки в Шымкенте
Допустим, в октябре в Шымкенте было бы столько же заказов и тот же средний чек оплаченных, а доля отмен осталась как в сентябре. Какой была бы выручка Шымкента в октябре? Сколько компания потеряла из-за отмен?
Сравните потерю с падением всей компании (823 890 ₸). Что было бы с выручкой компании без сбоя?

In [11]:
# 7. сколько стоил сбой доставки в Шымкенте
sh = df[df["city"] == "Шымкент"]
sh_sep, sh_oct = sh[sh["month"] == 9], sh[sh["month"] == 10]

n_oct = len(sh_oct)
check_oct = sh_oct.loc[sh_oct["status"] == "оплачен", "amount"].mean()
cancel_sep = sh_sep["cancel"].mean()

expected = n_oct * (1 - cancel_sep) * check_oct
actual = sh_oct.loc[sh_oct["status"] == "оплачен", "amount"].sum()
loss = expected - actual

print(f"заказов в октябре: {n_oct}, чек {check_oct:,.0f} ₸, отмены сентября {cancel_sep:.1%} (из {len(sh_sep)})")
print(f"выручка октября без сбоя: {expected:,.0f} ₸, на деле: {actual:,.0f} ₸")
print(f"потеря: {loss:,.0f} ₸, в {loss / 823_890:.1f} раза больше падения всей компании")

company = paid.groupby("month")["amount"].sum()
fixed_oct = company[10] + loss
print(f"\nкомпания: сентябрь {company[9]:,.0f}, октябрь {company[10]:,.0f}, без сбоя {fixed_oct:,.0f} ₸")
print(f"изменение без сбоя: {(fixed_oct / company[9] - 1) * 100:+.1f}%")

заказов в октябре: 417, чек 13,993 ₸, отмены сентября 8.9% (из 359)
выручка октября без сбоя: 5,315,006 ₸, на деле: 4,044,010 ₸
потеря: 1,270,996 ₸, в 1.5 раза больше падения всей компании

компания: сентябрь 35,243,410, октябрь 34,419,520, без сбоя 35,690,516 ₸
изменение без сбоя: +1.3%


**Ответ:** если бы в октябре в Шымкенте были те же 417 заказов и тот же чек 13 993 ₸, а отменяли бы, как в сентябре, 8,9%, выручка была бы около 5,32 млн ₸. На деле 4,04 млн, то есть сбой доставки стоил примерно 1,27 млн ₸.

Это в 1,5 раза больше, чем падение всей компании (823 890 ₸). Без сбоя компания в октябре не упала бы на 2,3%, а выросла бы примерно на 1,3%: 35,69 млн ₸ против 35,24 млн в сентябре.

**Менеджеру:** всё падение выручки и даже больше объясняется одной поломкой доставки в Шымкенте. Остальной бизнес в октябре рос.

## [Челлендж] 8. Органика −6,0%: сигнал или шум?
Разбейте клиентов из customers.csv **случайно** на три группы в тех же пропорциях, что и каналы (`np.random.default_rng(7)`, `rng.choice`, `map`). Посчитайте изменение выручки оплаченных окт/сент для каждой группы. Повторите 100 раз в цикле.
В каком диапазоне меняется выручка случайной группы размером с органику (5% и 95% перцентили)? Можно ли по «органика −6,0%» делать вывод о SEO?

In [12]:
# 8. органика -6%: сигнал или шум
import numpy as np

m_paid = m[m["status"] == "оплачен"]
real = m_paid.pivot_table(index="channel", columns="month", values="amount", aggfunc="sum")
print("настоящие каналы, изменение окт/сент, %:")
print(((real[10] / real[9] - 1) * 100).round(1))
print(cu["channel"].value_counts())

rng = np.random.default_rng(7)
channels = cu["channel"].to_numpy()
results = []
for _ in range(100):
    # replace=False перемешивает каналы между клиентами: размеры групп те же, связи с поведением нет
    fake = pd.Series(rng.choice(channels, size=len(channels), replace=False), index=cu["customer_id"])
    g = paid.assign(group=paid["customer_id"].map(fake))
    t = g.pivot_table(index="group", columns="month", values="amount", aggfunc="sum")
    results.append((t[10] / t[9] - 1) * 100)

res = pd.DataFrame(results)
org = res["органика"]
print(f"\nслучайная группа размером с органику: 5% {org.quantile(0.05):.1f}%, "
      f"медиана {org.median():.1f}%, 95% {org.quantile(0.95):.1f}%")
print(f"в скольких из 100 случайных разбиений было -6% и хуже: {(org <= -6).sum()}")

настоящие каналы, изменение окт/сент, %:
channel
органика   -6.0
реклама    -3.1
рефералы    9.4
dtype: float64
channel
реклама     1133
органика     743
рефералы     324
Name: count, dtype: int64

случайная группа размером с органику: 5% -9.9%, медиана -2.1%, 95% 6.2%
в скольких из 100 случайных разбиений было -6% и хуже: 19


**Ответ:** у настоящей органики -6,0%. Но если перемешать каналы между клиентами случайно, группа такого же размера (743 клиента) в 90% случаев меняется от -9,9% до +6,2%, а в 19 из 100 разбиений падает на 6% и сильнее. То есть -6% вполне укладывается в обычный разброс.

**Менеджеру:** по этой цифре нельзя делать вывод, что сломалось SEO. Такое падение получается и у случайной группы клиентов. К тому же на уроке было видно, что доля Шымкента во всех каналах примерно одинаковая, около 15%, так что все каналы задевает одна и та же проблема с доставкой.